# 四探针开源精简版

唯一测量入口。保留单点/9点/25点、自动量程、简单正反测量、色块热力图、CSV/JSON历史及安全恢复。电压按ADC原生单位换算，不额外乘4，不提供显示倍率。取消网页、自动图片上传及高级PL诊断。

必须整包上传并使用本版bit/hwh。屏幕材料4仍为GaN，沿用AlGaN模型；历史中的AlN不改名。无本轮板卡验证，默认安全授权关闭。探针带压依赖Z保持力时，禁止重载Overlay或重启Kernel。


## 1. 导入与版本检查


In [ ]:
from pathlib import Path
import sys, time, inspect
ROOT = Path.cwd()
# 交付版：在Four_ALL目录打开本Notebook；相关Python模块就在同一目录。
# 本地工程版：可复用模块位于sw/pynq。
MODULE_ROOT = ROOT if (ROOT/'hmi_four_all.py').is_file() else ROOT.parent/'pynq'
assert (MODULE_ROOT/'hmi_four_all.py').is_file(), 'Open the delivered Notebook directory or sw/notebooks'
sys.path.insert(0, str(MODULE_ROOT))
from pynq import Overlay, MMIO
from four_probe_sw.analog_frontend import AnalogFrontendController
from four_probe_sw.ads8681_frame_diagnostics import Ads8681FrameMonitor
from four_probe_sw.direct_adc_measurement import DIRECT_ADC_MEASUREMENT_VERSION, DEFAULT_PROBE_VOLTAGE_OFFSET_BY_RANGE_V
from four_probe_sw.final_four_probe_system import FinalFourProbeSystem, FINAL_FOUR_PROBE_SYSTEM_VERSION, check_deploy_manifest
from four_probe_sw.pl_autorange import PlAutorange, AutorangeConfig, PL_AUTORANGE_VERSION
from four_all_hmi_ui.hmi_address import NumericPageUart
from four_all_hmi_ui.uart import Uart
from hmi_four_all import FourAllHmiService, make_system_factory, FOUR_ALL_HMI_VERSION
assert DIRECT_ADC_MEASUREMENT_VERSION == 19
assert FOUR_ALL_HMI_VERSION == 11 and FINAL_FOUR_PROBE_SYSTEM_VERSION == 24, "Upload all new modules and Restart Kernel"
assert PL_AUTORANGE_VERSION == 5, "Upload all new modules and Restart Kernel"
print('HMI integration:', FOUR_ALL_HMI_VERSION, 'ADC version:', DIRECT_ADC_MEASUREMENT_VERSION)
print('SOURCE probe offsets V:', DEFAULT_PROBE_VOLTAGE_OFFSET_BY_RANGE_V)

from hmi_range04 import RangePictures, RangeFrontendProxy, RangePlProxy, RangeVisualService




## 2. 机械、采样与安全配置
修改后在安全停止状态重新执行配置和服务格，不必为改参数重载Overlay。


In [ ]:
# 安全授权：检查真实机构/回路后手动开启；默认不允许100mA。
ALLOW_MOTION = False
ARM_CURRENT = False
ALLOW_100MA = False
GEOMETRY_CONFIRMED = False
# 机械标定：步、毫米。不是本轮重新标定结果。
R_HOME_TO_CENTER_STEPS = 8000
R_RING_SPACING_STEPS = 3500
R_STEPS_PER_MM = 800.0
R_MAX_STEPS = 40000
X_PULSES_PER_REV = 12000
FULL_DOWN_STEPS = 7450
HALF_UP_STEPS = 3725
FULL_DOWN_STEPS_BY_MATERIAL = {'silicon':7450,'algan':7600,'hemt':7600,'gan':7600}
PROBE_MARGIN_MM = 1.5
# Z步进周期：50MHz时钟拍。大周期表示更慢；硬限位仍由PL保护。
Z_RUN_STEP_CYCLES = 6000
Z_END_STEP_CYCLES = 96000
Z_DECEL_WINDOW_STEPS = 500
Z_DECEL_SEGMENT_STEPS = 31
Z_DECEL_SEGMENTS = 16
Z_REHOME_INTERVAL_POINTS = 5
Z_TIMEOUT_S = 20.0
R_TIMEOUT_S = 20.0
X_TIMEOUT_S = 10.0
POLL_INTERVAL_S = .01
MOTION_TRACE = True
# PS回退路径采样参数：数量、秒、伏特；原生ADC换算，不附加增益。
RANGE_SELECTION_SAMPLES = 100
VALID_SAMPLES = 1000
SAMPLE_INTERVAL_S = .002
DISCARD_FIRST = 20
RANGE_SETTLE_S = 3.0
BIDIRECTIONAL_SETTLE_S = 3.0
ADC_CONFIG_SETTLE_S = .5
CAPTURE_TIMEOUT_S = 10.0
ADC_INPUT_VALID_MIN_V = .005
ADC_INPUT_VALID_MAX_V = 5.0
PROBE_ABS_MIN_V = .005
PROBE_ABS_MAX_V = 2.0
RANGE_INDICES = (0,1,2)
# 简单双极性试档和单对正反测量。不采关电流后的悬空节点作零位。
SELECT_RANGE_BY_RESISTIVITY = False
ZERO_OFFSET_SAMPLES = 0
ZERO_OFFSET_SETTLE_S = .5
MAX_ZERO_OFFSET_V = .05
PL_WINDOW_ABS_MIN_V = 0.0
PILOT_BIPOLAR = True
RESELECT_RANGE_EVERY_POINT = False
FULL_RANGE_RESELECT_ON_RETRY = True
CHECK_CURRENT_PATH = True
USE_PL_RHO = True
PL_RHO_TOLERANCE = 1e-4
# PL v5：基本均值判稳、选档、每方向一个积分窗口，无高级诊断。
USE_PL_AUTORANGE = True
PL_AUTORANGE_CONFIG = AutorangeConfig(pilot_window_ms=100,measure_window_ms=400,
                                      settle_max_ms=5000,warmup_ms=0)
PL_AUTORANGE_CONFIG.registers()
STOP_AFTER_COMPLETE = True
print('ADC19 / PS24 / HMI11 / FP_DSP5; native voltage; one F/R pair')


## 3. 加载配套Overlay
先停止其他服务并确认探针不依赖Z保持力。真实复位锁存必须人工恢复，不能靠重新加载bit清除。


In [ ]:
def find_overlay_pair(root):
    import xml.etree.ElementTree as ET
    candidates=[root/'all_design.bit',root/'build'/'latest_amd_overlay'/'all_design.bit']
    for bit in candidates:
        hwh=bit.with_suffix('.hwh')
        if not bit.is_file() or not hwh.is_file():continue
        modules={m.get('INSTANCE'):m for m in ET.parse(hwh).findall('.//MODULE')}
        if 'ALL_Final_0' not in modules or 'axi_uartlite_0' not in modules:continue
        params={p.get('NAME'):p.get('VALUE') for p in modules['axi_uartlite_0'].findall('.//PARAMETER')}
        if params.get('C_BAUDRATE')=='115200':return bit
    raise FileNotFoundError('Matching ALL_Final + UART115200 bit/hwh pair not found')
if 'mmio' in globals():
    if int(mmio.read(0x04)) & 0x30000:
        raise RuntimeError('真实PL复位/看门狗锁存：必须按人工安全流程处理，不能重配绕过')
    if (int(mmio.read(0x34)) | int(mmio.read(0x4C)) | int(mmio.read(0x70))) & 1:
        raise RuntimeError('轴仍忙，禁止重新配置PL')
    if 'frontend' in globals() and frontend.status().get('current_active',True):
        raise RuntimeError('电流仍开或状态不明，禁止重新配置PL')
if 'emergency' in globals() and emergency.stopped.is_set():
    raise RuntimeError('真实PS急停标志仍在，先按人工流程恢复')
from hmi_emergency import stop_before_overlay
stop_before_overlay(
    uart=globals().get('uart'),
    systems=(getattr(globals().get('service'), 'system', None), globals().get('system')),
)
if 'emergency' in globals() and emergency.stopped.is_set():
    raise RuntimeError('真实PS急停标志仍在，禁止重新配置PL')
if 'mmio' in globals():
    if int(mmio.read(0x04)) & 0x30000:
        raise RuntimeError('真实PL复位/看门狗锁存：禁止重新配置PL')
    if (int(mmio.read(0x34)) | int(mmio.read(0x4C)) | int(mmio.read(0x70))) & 1:
        raise RuntimeError('轴仍忙，禁止重新配置PL')
    if 'frontend' in globals() and frontend.status().get('current_active',True):
        raise RuntimeError('电流仍开或状态不明，禁止重新配置PL')
BIT=find_overlay_pair(ROOT)
check_deploy_manifest(MODULE_ROOT/'four_probe_sw'/'deploy_manifest.json',
                      MODULE_ROOT/'four_probe_sw', BIT)
overlay = Overlay(str(BIT), download=True)
assert 'ALL_Final_0' in overlay.ip_dict and 'axi_uartlite_0' in overlay.ip_dict, 'Wrong overlay: UART missing'
assert int(overlay.ip_dict['ALL_Final_0']['phys_addr']) == 0x40000000
assert int(overlay.ip_dict['axi_uartlite_0']['phys_addr']) == 0x42C00000
mmio = MMIO(0x40000000, 0x10000)
frontend = AnalogFrontendController(mmio, timeout_s=2.0)
frontend.safe_off()
assert mmio.read(0x10) == 0x414C5835
raw_uart = Uart(MMIO(0x42C00000, 0x10000))
raw_uart.reset()
raw_uart.send_command('bkcmd=0')
uart = NumericPageUart(raw_uart)
from hmi_emergency import EmergencyReceiver
emergency = EmergencyReceiver(mmio)
uart.rx_transform = emergency.feed
uart.rx_error_handler = emergency.reset_parser
uart.rx_fault_handler = emergency.request
uart.start_receiving(poll_s=.0002)
uart.trace = True
monitor = Ads8681FrameMonitor(mmio, sample_interval_s=SAMPLE_INTERVAL_S)
pl_autorange = None
if 'fp_dsp_0' in overlay.ip_dict:
    fp_dsp = PlAutorange.from_overlay(overlay, MMIO, PL_AUTORANGE_CONFIG)
    print('FP_DSP:', fp_dsp.identify(), fp_dsp.diagnostics())
    if USE_PL_AUTORANGE:
        pl_autorange = fp_dsp
    else:
        print('USE_PL_AUTORANGE=False：PS正反采集路径')
elif USE_PL_AUTORANGE:
    raise RuntimeError('USE_PL_AUTORANGE=True，但本位流不含FP_DSP：请换用含fp_dsp_0的bit/hwh，或改回False')
print('BIT:', BIT, 'state=0x%X'%mmio.read(0x04), 'R/Z/X positions:',*[mmio.read(r) for r in (0x38,0x50,0x74)])
print('Current is OFF. Do not arm until all mechanical/current checks are complete.')


## 4. 只读状态核对
本格不回零、不移动。


In [ ]:
print('Safety STATUS=0x%08X' % mmio.read(0x04))
print('Z/R position:',mmio.read(0x50),mmio.read(0x38))
frontend.safe_off()


## 5. 串口屏统一服务
本格保持运行后点击开始。只有一个UART所有者。RANGE_READY仅表示选档完成；完成必须以结果和保存状态为准。成功一场后正常退出；Ctrl-C触发复位保持，绝不自动回零。


In [ ]:
def check_range_reset():
    if emergency.stopped.is_set():raise RuntimeError('SYSTEM_RESET')
pictures=RangePictures(uart,abort_check=check_range_reset)
observed_frontend=RangeFrontendProxy(frontend,pictures,allowed=RANGE_INDICES)
observed_pl=RangePlProxy(pl_autorange,pictures) if pl_autorange is not None else None
factory = make_system_factory(
    mmio, observed_frontend, monitor,
    range_selection_sample_count=RANGE_SELECTION_SAMPLES, valid_sample_count=VALID_SAMPLES, sample_interval_s=SAMPLE_INTERVAL_S,
    discard_first=DISCARD_FIRST, range_settle_s=RANGE_SETTLE_S,
    bidirectional_settle_s=BIDIRECTIONAL_SETTLE_S, adc_config_settle_s=ADC_CONFIG_SETTLE_S,
    capture_timeout_s=CAPTURE_TIMEOUT_S, adc_input_min_v=ADC_INPUT_VALID_MIN_V, adc_input_max_v=ADC_INPUT_VALID_MAX_V,
    probe_abs_min_v=PROBE_ABS_MIN_V, probe_abs_max_v=PROBE_ABS_MAX_V,
    r_home_to_center_steps=R_HOME_TO_CENTER_STEPS, r_ring_spacing_steps=R_RING_SPACING_STEPS,
    r_max_steps=R_MAX_STEPS, x_pulses_per_rev=X_PULSES_PER_REV,
    full_down_steps=FULL_DOWN_STEPS, half_up_steps=HALF_UP_STEPS,
    full_down_steps_by_material=FULL_DOWN_STEPS_BY_MATERIAL,
    z_run_step_cycles=Z_RUN_STEP_CYCLES, z_end_step_cycles=Z_END_STEP_CYCLES,
    z_decel_window_steps=Z_DECEL_WINDOW_STEPS, z_decel_seg_steps=Z_DECEL_SEGMENT_STEPS,
    z_decel_segments=Z_DECEL_SEGMENTS, z_rehome_interval_points=Z_REHOME_INTERVAL_POINTS,
    z_timeout_s=Z_TIMEOUT_S, r_timeout_s=R_TIMEOUT_S, x_timeout_s=X_TIMEOUT_S,
    poll_interval_s=POLL_INTERVAL_S, motion_trace=MOTION_TRACE,
    select_range_by_resistivity=SELECT_RANGE_BY_RESISTIVITY,
    range_indices=RANGE_INDICES,allow_100ma=ALLOW_100MA,
    zero_offset_samples=ZERO_OFFSET_SAMPLES,zero_offset_settle_s=ZERO_OFFSET_SETTLE_S,
    max_zero_offset_v=MAX_ZERO_OFFSET_V,pl_window_abs_min_v=PL_WINDOW_ABS_MIN_V,
    pilot_bipolar=PILOT_BIPOLAR,check_current_path=CHECK_CURRENT_PATH,
    reselect_range_every_point=RESELECT_RANGE_EVERY_POINT,
    full_range_reselect_on_retry=FULL_RANGE_RESELECT_ON_RETRY,
    use_pl_window_stats=True,pl_stats_timeout_s=.1,pl_stats_min_valid_samples=100,
    use_pl_calc=USE_PL_RHO,pl_rho_tolerance=PL_RHO_TOLERANCE,bitstream_path=str(BIT),
    pl_autorange=observed_pl,
)
service = RangeVisualService(
    uart, observed_frontend, factory,
    pictures=pictures, allow_motion=ALLOW_MOTION, arm_current=ARM_CURRENT,
    allow_100ma=ALLOW_100MA, geometry_confirmed=GEOMETRY_CONFIRMED, enabled_ranges=RANGE_INDICES,
    r_steps_per_mm=R_STEPS_PER_MM, r_ring_spacing_steps=R_RING_SPACING_STEPS,
    probe_margin_mm=PROBE_MARGIN_MM, emergency=emergency, output_dir=ROOT/'four_all_results', trace=True,
)
if (ROOT/'STOP_HMI_LISTENER').exists():
    raise RuntimeError('存在STOP_HMI_LISTENER：监听已人工停止；处理完后移除该标记再启动')
uart.stop_receiving()
raw_uart.reset()
raw_uart.send_command('bkcmd=0')
uart.discard_pending_rx()
uart.start_receiving(poll_s=.0002)
print('[FOUR_ALL_HMI] LISTEN: START single/25/9; HIST slots 1..4; one UART owner')
print('Complete real history:', service.history.list_history())
stop_listener=False
exit_reason='unexpected exception'
try:
    while True:
        if (ROOT/'STOP_HMI_LISTENER').exists():
            exit_reason='manual listener stop file'
            break
        status=int(mmio.read(0x04))
        if status & 0x30000:
            exit_reason='PL watchdog latched' if status & 0x20000 else 'PL reset latched'
            break
        if uart.rx_fatal_error:
            emergency.request('RX pump stopped')
            exit_reason='RX pump stopped: '+uart.rx_fatal_error
            break
        if emergency.stopped.is_set():
            exit_reason='SYSTEM_RESET requested'
            print('SYSTEM_RESET latched; axes held. Manual CLEAR and rehome required.')
            break
        for result in service.poll_once():
            print('[FOUR_ALL_HMI] RESULT', result)
            if result['status'] == 'HISTORY_ERROR':
                print('History unavailable or invalid; no scan started:', result['error'])
                continue
            if result['status']=='COMPLETE' and STOP_AFTER_COMPLETE:
                stop_listener=True
                exit_reason='COMPLETE (STOP_AFTER_COMPLETE=True)'
            if result['status'] not in ('COMPLETE', 'RESTORED'):
                print('[FOUR_ALL_HMI] keep LISTEN; no automatic retry:',result.get('error'))
        status=int(mmio.read(0x04))
        if status & 0x30000 or emergency.stopped.is_set():
            exit_reason=('PL watchdog latched' if status & 0x20000 else 'PL reset latched' if status & 0x10000 else 'SYSTEM_RESET requested')
            break
        if stop_listener:break
        time.sleep(.001)
except KeyboardInterrupt:
    exit_reason='KeyboardInterrupt (SYSTEM_RESET)'
    emergency.request('Ctrl-C')
    print('SYSTEM_RESET requested; axes held, no automatic HOME')
except Exception as exc:
    exit_reason='unexpected exception: '+str(exc)
    raise
finally:
    print('[FOUR_ALL_HMI] EXIT reason='+exit_reason)
    if pl_autorange is not None:
        pl_autorange.abort()  # FP_DSP停止并关电流；ALL_Final的safe_off随后再关一次
    frontend.safe_off()
    if service.system is not None:service.system.finish_safety_cleanup()
    print('Current OFF; UART RX remains active for RST. Stop RX before reloading the Overlay; inspect axes before touching.')


## 6. 人工机械恢复
服务已退出、确认机构安全后才允许执行。会真实移动Z/R，X角度需人工确认。


In [ ]:
# 仅在测量服务格已经退出、人工检查机构/限位/间隙并确认电流关闭后运行。
# 本格会真实移动Z/R至HOME；X不动。保留原唯一RX，RST仍可中止恢复。
MANUAL_CLEAR_AND_REHOME = False
if MANUAL_CLEAR_AND_REHOME:
    from hmi_recovery import MechanicalRecoveryController, manual_clear_and_rehome
    _previous_systems=(getattr(globals().get('service'),'system',None),
                       globals().get('system'),globals().get('recovery_system'))
    # 只传机械参数，不调用测量factory，不依赖屏幕材料、膜厚或GaN确认状态。
    _recovery_down=FULL_DOWN_STEPS
    _recovery_half=HALF_UP_STEPS
    recovery_system=MechanicalRecoveryController(
        mmio,frontend,
        r_home_to_center_steps=R_HOME_TO_CENTER_STEPS,r_ring_spacing_steps=R_RING_SPACING_STEPS,
        r_max_steps=R_MAX_STEPS,x_pulses_per_rev=X_PULSES_PER_REV,
        full_down_steps=_recovery_down,half_up_steps=_recovery_half,
        z_run_step_cycles=Z_RUN_STEP_CYCLES,z_end_step_cycles=Z_END_STEP_CYCLES,
        z_decel_window_steps=Z_DECEL_WINDOW_STEPS,z_decel_seg_steps=Z_DECEL_SEGMENT_STEPS,
        z_decel_segments=Z_DECEL_SEGMENTS,z_rehome_interval_points=Z_REHOME_INTERVAL_POINTS,
        z_timeout_s=Z_TIMEOUT_S,r_timeout_s=R_TIMEOUT_S,x_timeout_s=X_TIMEOUT_S,
        poll_interval_s=POLL_INTERVAL_S,motion_trace=MOTION_TRACE)
    manual_clear_and_rehome(
        recovery_system,uart,emergency,confirmed=True,previous_systems=_previous_systems,
        pl_autorange=globals().get('pl_autorange'))
    system=recovery_system
    if 'service' in globals():
        service.system=recovery_system
    print('Z/R HOME verified. Re-execute service cell for a new scan. X angle must be checked manually.')
else:
    print('Locked: no clearing, no recovery movement')


## 7. 可选原始串口抓包
仅安全静止时使用。抓到的START丢弃，RST仍生效。


In [ ]:
# 只旁听屏幕→PS，不发送任何串口命令，不启动测量或运动。
# 必须等真实服务格正常退出后执行；运行时不要启动其他Notebook或串口服务。
CAPTURE_SCREEN_INPUT = False  # 确认安全静止后手动改True
CAPTURE_SECONDS = 30.0
if not CAPTURE_SCREEN_INPUT:
    raise RuntimeError('抓包锁定：确认机构安全、电流关闭、服务格已退出，再设True')
import time, json
from pathlib import Path
from collections import deque
if not 0 < float(CAPTURE_SECONDS) <= 300:
    raise ValueError('抓包时间必须为0..300秒')
if (int(mmio.read(0x34)) | int(mmio.read(0x4C)) | int(mmio.read(0x70))) & 1:
    raise RuntimeError('轴仍忙，不能抓包')
if frontend.status().get('current_active', True):
    raise RuntimeError('未确认电流关闭，不能抓包')
if uart._rx_thread is None or not uart._rx_thread.is_alive():
    raise RuntimeError('原唯一RX线程未运行；先按01正常流程初始化，不能新建第二个读者')
if uart.rx_fatal_error:
    raise RuntimeError('RX线程故障：'+str(uart.rx_fatal_error))
previous_transform = uart.rx_transform
if previous_transform is None:
    raise RuntimeError('缺少原RST处理器，禁止绕过急停接收')
pending = deque(maxlen=2048)
records = []
capture_stats = {'dropped_chunks': 0}
def capture_tap(data):
    # RX线程只复制，不在这里打印/解码/写盘；仍调用原始RST过滤器。
    if len(pending) == pending.maxlen:
        capture_stats['dropped_chunks'] += 1
    pending.append((time.monotonic(), bytes(data)))
    return previous_transform(data)
def show_input(timestamp, data):
    records.append({'monotonic_s': timestamp, 'hex': data.hex(' ')})
    print('RX HEX:', data.hex(' '))
    if all(b in (9,10,13) or 32 <= b < 127 for b in data):
        print('RX ASCII:', repr(data.decode('ascii')))
    else:
        for encoding in ('utf-8', 'gb18030'):
            try:
                decoded = data.decode(encoding)
            except UnicodeDecodeError:
                continue
            if all(c.isprintable() or c in '\r\n\t' for c in decoded):
                print('RX候选文字（'+encoding+'，需核对屏幕编码）:', repr(decoded))
                break
        else:
            print('二进制/不完整文字片段，不直接当作文本；以HEX为准')
print('开始旁听30秒（或配置时长）。仅打印原始输入，不调用service.poll_once；START不会在本格启动测量。')
print('屏幕键盘修改变量不会自动通知PS，屏端必须用prints/printh发送。RST仍会实际触发复位。')
started = time.monotonic()
uart.rx_transform = capture_tap
try:
    while time.monotonic()-started < CAPTURE_SECONDS:
        while pending:
            show_input(*pending.popleft())
        if uart.rx_fatal_error:
            raise RuntimeError(uart.rx_fatal_error)
        time.sleep(.01)
except KeyboardInterrupt:
    print('仅结束静止抓包；未清除或修改任何PL故障状态')
finally:
    if uart.rx_transform is capture_tap:
        uart.rx_transform = previous_transform
    while pending:
        show_input(*pending.popleft())
    # 不让抓到的测试START在下次运行服务时被当作新请求执行。
    uart.discard_pending_rx()
    if 'service' in globals():
        from hmi_four_all import HistoryWireParser
        service.parser = HistoryWireParser()
    if hasattr(emergency, 'reset_parser'):
        emergency.reset_parser()  # 只清残片，绝不清stopped或PL锁存。
    folder = Path(ROOT)/'uart_capture'
    folder.mkdir(exist_ok=True)
    path = folder/('screen_rx_'+time.strftime('%Y%m%d_%H%M%S')+'_'+str(time.time_ns())+'.json')
    payload = {'direction':'screen_to_ps', 'chunks': records, **capture_stats,
               'note':'RX线程分块不是协议帧边界；候选文字不是已验证协议解释'}
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    raw_input = bytes.fromhex(' '.join(r['hex'] for r in records))
    print('全部输入 HEX:', raw_input.hex(' ') if raw_input else '未收到数据')
    for encoding in ('utf-8','gb18030'):
        try:
            text_input = raw_input.decode(encoding)
        except UnicodeDecodeError:
            continue
        if raw_input and all(c.isprintable() or c in '\r\n\t' for c in text_input):
            print('完整候选文字（'+encoding+'）:', repr(text_input))
            break
    if b'RST\r\n' in raw_input:
        print('包含RST复位请求；原急停处理器仍生效，请检查实际锁存')
    print('丢弃抓包块数:', capture_stats['dropped_chunks'], '抓包保存:', path)
    print('已恢复原RX处理器；没有数据不代表按键没按，需确认屏端发送事件。')
